# Phase 5: Agent Evaluation

Measure Faithfulness, Task Success Rate, Latency, and Cost.

In [ ]:
import sys
sys.path.insert(0, '..')

## 5.1 Single Query Evaluation

In [ ]:
import time
from src.rag.chain import query_rag
from src.evaluation.evaluator import run_full_evaluation

# Run a query and measure latency
question = 'Does COVID-19 qualify as force majeure under a supply contract?'
start = time.time()
result = query_rag(question, doc_type='case_law')
latency = time.time() - start

# Get context from source documents
context = '\n\n'.join(doc.page_content for doc in result.get('source_documents', []))

# Run full evaluation
evaluation = run_full_evaluation(
    question=question,
    answer=result['answer'],
    context=context,
    expected_topics=['Force majeure clause interpretation', 'COVID-19 applicability', 'Ejusdem generis doctrine'],
    latency=latency,
)

print('Evaluation Summary:')
for key, value in evaluation['summary'].items():
    print(f'  {key}: {value}')

## 5.2 Full Benchmark (10 Test Cases)

In [ ]:
from src.evaluation.benchmark import run_benchmark

report = run_benchmark(
    output_file='../evaluation_report/benchmark_results.json'
)

print('\nAggregate Results:')
for key, value in report['aggregate'].items():
    print(f'  {key}: {value}')

## 5.3 Cross-Provider Comparison

In [ ]:
from src.config import get_available_providers

providers = get_available_providers()
print(f'Available providers for comparison: {providers}')

# Run benchmark for each provider
all_reports = {}
for provider in providers:
    print(f'\n--- Running benchmark with {provider} ---')
    report = run_benchmark(
        llm_provider=provider,
        output_file=f'../evaluation_report/benchmark_{provider}.json'
    )
    all_reports[provider] = report

# Compare results
print('\n=== CROSS-PROVIDER COMPARISON ===')
for provider, report in all_reports.items():
    agg = report.get('aggregate', {})
    print(f'\n{provider.upper()}:')
    print(f'  Faithfulness: {agg.get("avg_faithfulness", "N/A")}')
    print(f'  Success Rate: {agg.get("avg_success_rate", "N/A")}')
    print(f'  Avg Latency:  {agg.get("avg_latency", "N/A")}s')
    print(f'  Total Cost:   ${agg.get("total_cost", "N/A")}')